# Solar Panel Model Comparison & Temporal Growth Analysis (2021-2025)
This notebook:
1. Compares test metrics between **YOLO11m-seg** and **Mask R-CNN (ResNet-50 FPN)**.
2. Runs temporal inference across all years (2021 to 2025).
3. Generates growth visualization charts and adoption metrics.

## 1. Setup Environment

In [ ]:
!pip install -q ultralytics matplotlib numpy

## 2. Check Results & Run Comparison

In [ ]:
import json
import csv
from pathlib import Path

print("=" * 70)
print("MODEL COMPARISON: YOLO11m-seg vs Mask R-CNN (ResNet-50 FPN)")
print("=" * 70)

# YOLO metrics
yolo_results = Path("runs/solar-seg/yolo11m-seg/results.csv")
if yolo_results.exists():
    with open(yolo_results) as f:
        rows = list(csv.DictReader(f))
        if rows:
            print("\nYOLO11m-seg Best/Last Epoch:")
            for k, v in rows[-1].items():
                k = k.strip()
                if "map" in k.lower() or "loss" in k.lower():
                    print(f"  {k}: {float(v.strip()):.4f}")
else:
    print("\n[!] YOLO results not found yet.")

# Mask R-CNN metrics
mask_results = Path("runs/solar-seg/mask-rcnn/metrics.json")
if mask_results.exists():
    data = json.loads(mask_results.read_text())
    print("\nMask R-CNN Final Metrics:")
    for k, v in data.items():
        if "segm" in k or "bbox" in k:
            print(f"  {k}: {v:.4f}")
else:
    print("\n[!] Mask R-CNN results not found yet.")

## 3. Run Temporal Growth Analysis (2021 - 2025)
Runs predictions with the best YOLO weights across each year's imagery and plots growth trends.

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

model_path = Path("runs/solar-seg/yolo11m-seg/weights/best.pt")
if not model_path.exists():
    print("Waiting for model weights at:", model_path)
else:
    model = YOLO(str(model_path))
    years = ["2021", "2022", "2023", "2024", "2025"]
    yolo_root = Path("solar-panel-exports/YOLO")
    output_dir = Path("runs/solar-seg/temporal-analysis")
    output_dir.mkdir(parents=True, exist_ok=True)

    yearly_stats = {}
    base_stats = defaultdict(lambda: defaultdict(dict))

    for year in years:
        panel_counts = []
        total_area = 0
        images_with_panels = 0
        total_images = 0

        for split in ["train", "valid", "test"]:
            img_dir = yolo_root / year / split / "images"
            if not img_dir.exists():
                continue

            results = model.predict(source=str(img_dir), imgsz=432, conf=0.25, save=False, verbose=False)
            for r in results:
                total_images += 1
                n_panels = len(r.masks.data) if r.masks is not None else 0
                panel_counts.append(n_panels)
                if n_panels > 0:
                    images_with_panels += 1
                    total_area += r.masks.data.sum().item()
                    fname = Path(r.path).stem
                    parts = fname.split("_")
                    for p in parts:
                        if p.startswith("Base"):
                            base_stats[p][year][parts[0]] = n_panels
                            break

        yearly_stats[year] = {
            "total_images": total_images,
            "images_with_panels": images_with_panels,
            "total_panels_detected": sum(panel_counts),
            "avg_panels_per_image": np.mean(panel_counts) if panel_counts else 0,
            "max_panels_in_image": max(panel_counts) if panel_counts else 0,
            "total_pixel_area": total_area,
            "adoption_rate": images_with_panels / total_images if total_images else 0,
        }

    # --- Visualization ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Solar Panel Growth Analysis (2021-2025)", fontsize=16, fontweight="bold")
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]

    # 1. Total Panels
    ax = axes[0, 0]
    counts = [yearly_stats[y]["total_panels_detected"] for y in years]
    bars = ax.bar(years, counts, color=colors, edgecolor="white", linewidth=1.5)
    ax.set_title("Total Solar Panels Detected", fontweight="bold")
    for b, c in zip(bars, counts):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 5, str(c), ha="center", va="bottom", fontweight="bold")

    # 2. Adoption Rate
    ax = axes[0, 1]
    rates = [yearly_stats[y]["adoption_rate"] * 100 for y in years]
    ax.plot(years, rates, "o-", color="#E91E63", linewidth=2.5, markersize=8)
    ax.fill_between(years, rates, alpha=0.15, color="#E91E63")
    ax.set_title("Solar Adoption Rate (%)", fontweight="bold")
    ax.set_ylim(0, 100)

    # 3. Avg Panels
    ax = axes[1, 0]
    avgs = [yearly_stats[y]["avg_panels_per_image"] for y in years]
    ax.plot(years, avgs, "s-", color="#4CAF50", linewidth=2.5, markersize=8)
    ax.fill_between(years, avgs, alpha=0.15, color="#4CAF50")
    ax.set_title("Avg Panels per Image", fontweight="bold")

    # 4. Heatmap by Base Location
    ax = axes[1, 1]
    base_names = sorted(base_stats.keys(), key=lambda x: int(x.replace("Base", "")) if x.replace("Base", "").isdigit() else 0)
    heatmap_data = [[sum(base_stats[b].get(y, {}).values()) for y in years] for b in base_names]
    if heatmap_data:
        im = ax.imshow(heatmap_data, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(len(years)))
        ax.set_xticklabels(years)
        ax.set_yticks(range(len(base_names)))
        ax.set_yticklabels(base_names, fontsize=8)
        fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title("Panels per Base Location", fontweight="bold")

    plt.tight_layout()
    plt.savefig(output_dir / "solar_growth_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    (output_dir / "yearly_stats.json").write_text(json.dumps(yearly_stats, indent=2))
    print("Saved chart and statistics successfully!")